# Klasifikasi DemogPairs Menggunakan ViT (Wajah, Emosi, dan Umur) & Logistic Regression

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(emotion_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

  0%|          | 0/10800 [00:00<?, ?it/s]

  5%|▌         | 552/10800 [00:00<00:01, 5511.48it/s]

 10%|█         | 1104/10800 [00:00<00:01, 5248.27it/s]

 15%|█▌        | 1641/10800 [00:00<00:01, 5300.51it/s]

 20%|██        | 2183/10800 [00:00<00:01, 5345.87it/s]

...

 89%|████████▉ | 9610/10800 [00:01<00:00, 5266.69it/s]

 94%|█████████▍| 10153/10800 [00:01<00:00, 5313.96it/s]

 99%|█████████▉| 10685/10800 [00:02<00:00, 5163.01it/s]

100%|██████████| 10800/10800 [00:02<00:00, 5198.54it/s]

Jumlah fitur per gambar: 2304


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [LogisticRegression(random_state=42)],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__max_iter': [500, 1000],
        'classifier__solver': ['lbfgs', 'saga'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

LogisticRegression: 96 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_lr_vit-face-emotion-age_",
    results_path="results/demogpairs_lr_vit-face-emotion-age_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: LogisticRegression


{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': None}


Accuracy  : 0.9273148148148148
Precision : 0.9274897245595951
Recall    : 0.927314814814815
F1 Score  : 0.9273371180995097
               precision    recall  f1-score   support

Asian_Females     0.9162    0.9111    0.9136       360
  Asian_Males     0.9076    0.9278    0.9176       360
Black_Females     0.9213    0.9111    0.9162       360
  Black_Males     0.9571    0.9306    0.9437       360
White_Females     0.9121    0.9222    0.9171       360
  White_Males     0.9505    0.9611    0.9558       360

     accuracy                         0.9273      2160
    macro avg     0.9275    0.9273    0.9273      2160
 weighted avg     0.9275    0.9273    0.9273      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9712962962962963,0.9162011173184358,0.9111111111111111,0.9136490250696379,360
Asian_Males,0.9722222222222222,0.907608695652174,0.9277777777777778,0.9175824175824175,360
Black_Females,0.9722222222222222,0.9213483146067416,0.9111111111111111,0.9162011173184358,360
Black_Males,0.9814814814814815,0.9571428571428572,0.9305555555555556,0.943661971830986,360
White_Females,0.9722222222222222,0.9120879120879121,0.9222222222222223,0.9171270718232044,360
White_Males,0.9851851851851852,0.9505494505494505,0.9611111111111111,0.9558011049723757,360


Confusion matrix saved: images\cm_lr_vit-face-emotion-age_LogisticRegression.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               328                13                 8                 0                11                 0
         Asian_Males                11               334                 2                 6                 0                 7
       Black_Females                 8                 3               328                 5                15                 1
         Black_Males                 0                11                 7               335                 1                 6
       White_Females                11                 1                11                 1               332                 4
         White_Males                 0                 6                 0                 3                 5               346


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
LogisticRegression,models/clf_demogpairs_lr_vit-face-emotion-age_LogisticRegression.pkl,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': None}",0.9273148148148148,0.9273371180995097,0.9274897245595951,0.927314814814815,270


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_lr_vit-face-emotion-age_LogisticRegression.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 12358.0,
 'days': 0,
 'hours': 3,
 'minutes': 25,
 'seconds': 58.0,
 'text': '0 hari 3 jam 25 menit 58.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 141643.0,
 'days': 1,
 'hours': 15,
 'minutes': 20,
 'seconds': 43.0,
 'text': '1 hari 15 jam 20 menit 43.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 2000, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': None}",0.9282,0.9219,0.9155,0.9219,0.9219,0.9219,0.9219,0.9222,0.9219,62.5325
2,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': None}",0.9282,0.9219,0.9155,0.9219,0.9219,0.9219,0.9219,0.9222,0.9219,47.8648
3,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'newton-cg', 'pca': None, 'scaler': None}",0.9282,0.9219,0.9155,0.9219,0.9219,0.9219,0.9219,0.9222,0.9219,58.1969
4,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 1000, 'classifier__solver': 'saga', 'pca': None, 'scaler': None}",0.9282,0.9213,0.9161,0.9213,0.9213,0.9216,0.9217,0.922,0.9216,207.0453
...,...,...,...,...,...,...,...,...,...,...,...
267,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': None}",0.8814,0.8692,0.864,0.8837,0.8733,0.8743,0.874,0.8747,0.8743,18.296
268,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'saga', 'pca': 'PCA', 'scaler': None}",0.8814,0.8692,0.864,0.8837,0.8733,0.8743,0.874,0.8747,0.8743,41.9455
269,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': None}",0.8814,0.8692,0.864,0.8837,0.8733,0.8743,0.874,0.8747,0.8743,13.8977
270,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': None}",0.8814,0.8692,0.864,0.8837,0.8733,0.8743,0.874,0.8747,0.8743,11.4181
